# 🦴 무릎 3D MRI 로부터 pseudo X-ray 만들기 (고등학생용 실습)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhj0326/knee-pseudo-xray-demo/blob/main/knee_pseudo_xray.ipynb)

이 노트북에서는 **무릎 3D MRI(자기공명영상)** 데이터로부터 **가짜 X-ray(pseudo X-ray)** 를
만드는 과정을 **단계별 이미지**로 직접 확인합니다. 그리고 같은 환자의 **실제 X-ray** 와 비교합니다.

### 무엇을 보게 되나요?
1. **3D MRI** — 무릎 속 뼈와 연골을 3차원으로 나눠 놓은 데이터
2. **pseudo X-ray** — 그 3D 데이터를 빛(X선)이 통과하듯 눌러서 만든 2차원 사진
3. **실제 X-ray** — 병원에서 찍은 진짜 X-ray 사진

> 💡 **핵심 아이디어**: X-ray 사진은 몸을 통과한 X선이 얼마나 약해졌는지를 찍은 것입니다.
> 뼈처럼 단단한 곳은 X선을 많이 막아 **하얗게**, 살처럼 무른 곳은 적게 막아 **어둡게** 보입니다.
> 우리는 3D 데이터를 한 방향으로 쭉 **더해서(투영해서)** 이 효과를 흉내 냅니다.

실행 방법: 위 메뉴에서 **런타임 → 모두 실행** 을 누르거나, 각 셀의 ▶ 버튼을 위에서부터 차례로 누르세요.

## 0단계. 데이터 내려받기 ⬇️

먼저 실습에 쓸 무릎 데이터(10명분)를 GitHub에서 내려받습니다.

> ⚠️ **아래 `GITHUB_USER` 를 선생님이 알려준 깃허브 아이디로 바꾼 뒤 실행하세요.**

In [ ]:
import os

# ↓↓↓ 여기를 본인(또는 선생님)의 깃허브 아이디로 바꾸세요 ↓↓↓
GITHUB_USER = "mhj0326"
REPO        = "knee-pseudo-xray-demo"
# ↑↑↑ ----------------------------------------------- ↑↑↑

# 이미 data 폴더가 있으면(=로컬 실행) 그대로 사용, 없으면 GitHub에서 clone
if os.path.exists("data/cases"):
    DATA = "data/cases"
elif os.path.exists(f"{REPO}/data/cases"):
    DATA = f"{REPO}/data/cases"
else:
    os.system(f"git clone https://github.com/{GITHUB_USER}/{REPO}.git")
    DATA = f"{REPO}/data/cases"

cases = sorted(os.listdir(DATA))
print("내려받은 환자 수:", len(cases))
print("환자 목록:", cases)

## 1단계. 필요한 도구 준비 🧰

계산(numpy), 그림 그리기(matplotlib), 3D 회전(scipy), 이미지 읽기(PIL) 도구를 불러옵니다.
코랩에는 모두 기본 설치되어 있어 추가 설치가 필요 없습니다.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import rotate as nd_rotate
from PIL import Image

# 한글 폰트 설정 (코랩 그래프 제목의 한글이 네모(□)로 깨지지 않게)
try:
    import matplotlib.font_manager as fm
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
        fm._load_fontmanager(try_read_cache=False)
    plt.rcParams["font.family"] = "NanumGothic"
except Exception as e:
    print("한글 폰트 설정은 건너뜁니다(제목이 영어로 보일 수 있어요):", e)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

# 뼈/연골 이름 (3D MRI 라벨 0~7 의 의미, 대략적인 구분)
CLASS_NAMES = {
    0: "배경(공기)",
    1: "뼈 1 (대퇴골)",
    2: "뼈 2 (경골)",
    3: "뼈 3 (슬개골)",
    4: "연골/무른조직 4",
    5: "연골/무른조직 5",
    6: "연골/무른조직 6",
    7: "연골/무른조직 7",
}

def load_case(name):
    """한 환자의 3D MRI 라벨 볼륨과 실제 X-ray 이미지를 불러온다."""
    folder = os.path.join(DATA, name)
    vol = np.load(os.path.join(folder, "mri_label.npz"))["vol"]   # (128,128,128) 값 0~7
    real = np.array(Image.open(os.path.join(folder, "real_xray.png")).convert("L"))
    return vol, real

print("도구 준비 완료! ✅")

## 2단계. 실습할 환자 1명 고르기 🧑‍⚕️

10명 중 한 명을 고릅니다. `PICK` 숫자(0~9)를 바꾸면 다른 환자로 실습할 수 있어요.

In [ ]:
PICK = 0                      # 0 ~ 9 사이 숫자로 바꿔보세요
CASE = cases[PICK]
vol, real = load_case(CASE)

print("선택한 환자:", CASE)
print("3D MRI 크기 :", vol.shape, "  (가로×세로×깊이 = 128×128×128 칸)")
print("들어있는 조직 번호:", np.unique(vol))

## 3단계. 3D MRI 들여다보기 🧊

3D MRI는 가로·세로·**깊이**가 있는 '정육면체 데이터'입니다.
한 번에 보기 어려우니, 세 방향으로 **잘라서(단면)** 봅니다.
각 조직(뼈·연골)은 서로 다른 색으로 칠했습니다.

In [ ]:
mid = vol.shape[0] // 2   # 한가운데 단면

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
views = [
    ("앞에서 본 단면 (관상면)", vol[:, mid, :]),
    ("옆에서 본 단면 (시상면)", vol[:, :, mid]),
    ("위에서 본 단면 (축상면)", vol[mid, :, :]),
]
for ax, (title, sl) in zip(axes, views):
    ax.imshow(np.flipud(sl), cmap="nipy_spectral", vmin=0, vmax=7, interpolation="nearest")
    ax.set_title(title); ax.axis("off")
plt.suptitle(f"환자 {CASE} 의 3D MRI 단면 (색 = 조직 종류)", fontsize=13)
plt.tight_layout(); plt.show()

print("조직 번호와 이름:")
for k in np.unique(vol):
    print(f"  {k} = {CLASS_NAMES[int(k)]}")

## 4단계. X선 감쇠(막는 정도) 정하기 🔦

X선이 조직을 통과할 때, **단단한 뼈는 많이 막고(값 큼)**, **무른 연골은 적게 막습니다(값 작음)**.
배경(공기)은 전혀 막지 않습니다(0). 이 '막는 정도'를 숫자로 정해 줍니다.

In [ ]:
# 조직 번호(0~7) 별로 X선을 막는 정도(감쇠계수)
mu = np.array([0.0,   # 0 배경
               1.0,   # 1 뼈
               1.0,   # 2 뼈
               1.0,   # 3 뼈
               0.35,  # 4 연골
               0.35,  # 5 연골
               0.35,  # 6 연골
               0.35]) # 7 연골

plt.figure(figsize=(7, 3))
plt.bar([CLASS_NAMES[i] for i in range(8)], mu, color="steelblue")
plt.ylabel("X선을 막는 정도"); plt.xticks(rotation=30, ha="right")
plt.title("조직별 X선 감쇠계수 (뼈=많이 막음, 연골=조금, 공기=0)")
plt.tight_layout(); plt.show()

## 5단계. 투영해서 pseudo X-ray 만들기 ✨ (가장 중요!)

이제 X선이 환자 몸을 **앞→뒤로 뚫고 지나간다**고 상상합니다.
지나가는 길에 만난 조직들의 '막는 정도'를 **전부 더하면**, 그 방향으로 얼마나 막혔는지가 나옵니다.
많이 막힌 곳(뼈)은 **밝게**, 적게 막힌 곳은 **어둡게** 표시하면 → **pseudo X-ray** 완성!

$$\text{픽셀 밝기} = \sum_{\text{앞}\to\text{뒤}} (\text{그 자리 조직이 X선을 막는 정도})$$

In [ ]:
def make_pseudo_xray(vol, mu, angle=0.0):
    """3D 볼륨을 angle 도 만큼 돌린 뒤, 앞->뒤 방향으로 더해 pseudo X-ray 생성."""
    dens = mu[vol]                                   # 각 칸을 '막는 정도'로 바꿈
    if angle != 0:
        dens = nd_rotate(dens, angle, axes=(1, 2),   # 서 있는 축을 기준으로 회전
                         reshape=False, order=1)
    pxray = dens.sum(axis=1)                          # 앞->뒤(axis=1)로 모두 더함
    return np.flipud(pxray)                           # 머리쪽이 위로 오게 뒤집기

pxray0 = make_pseudo_xray(vol, mu, angle=0)

plt.figure(figsize=(4, 6))
plt.imshow(pxray0, cmap="gray")
plt.title(f"환자 {CASE} 의 pseudo X-ray (정면, 0°)"); plt.axis("off")
plt.show()
print("밝은 곳 = 뼈(X선 많이 막음), 어두운 곳 = 무른 조직")

## 6단계. 여러 각도에서 바라보기 🔄

환자를 조금씩 돌려 가며 투영하면, 보는 각도에 따라 X-ray가 어떻게 달라지는지 알 수 있습니다.
(실제 촬영에서도 다리가 살짝 돌아가 있으면 사진이 달라집니다.)

In [ ]:
angles = [-20, -10, 0, 10, 20]
fig, axes = plt.subplots(1, len(angles), figsize=(13, 5))
for ax, a in zip(axes, angles):
    ax.imshow(make_pseudo_xray(vol, mu, angle=a), cmap="gray")
    ax.set_title(f"{a}°"); ax.axis("off")
plt.suptitle(f"환자 {CASE} — 보는 각도별 pseudo X-ray", fontsize=13)
plt.tight_layout(); plt.show()

## 7단계. 진짜 X-ray처럼 보정하기 🎚️

투영한 그대로는 밋밋합니다. 밝기 범위를 0~1로 맞추고(정규화), **감마 보정**으로 대비를 올리면
실제 X-ray 느낌에 더 가까워집니다.

In [ ]:
def normalize(img):
    img = img.astype(np.float32)
    return (img - img.min()) / (img.max() - img.min() + 1e-8)

raw   = normalize(pxray0)
gamma = raw ** 0.6            # 감마 < 1 : 밝은 부분(뼈) 대비 강조

fig, axes = plt.subplots(1, 2, figsize=(8, 6))
axes[0].imshow(raw,   cmap="gray"); axes[0].set_title("보정 전");       axes[0].axis("off")
axes[1].imshow(gamma, cmap="gray"); axes[1].set_title("보정 후 (감마)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 8단계. 실제 X-ray와 나란히 비교하기 👀

우리가 3D MRI로 만든 **pseudo X-ray** 와, 같은 환자의 **실제 X-ray** 를 비교해 봅니다.
완전히 똑같진 않지만, 뼈의 모양과 관절 위치가 닮은 것을 확인할 수 있어요.

> 참고: 실제 X-ray는 양쪽 다리 전체를 찍은 사진이라 가운데 자(ruler)가 보이고,
> pseudo X-ray는 한쪽 무릎의 3D 데이터만으로 만든 것입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 6))
axes[0].imshow(gamma, cmap="gray")
axes[0].set_title("우리가 만든 pseudo X-ray"); axes[0].axis("off")
axes[1].imshow(real, cmap="gray")
axes[1].set_title("실제 X-ray (병원 촬영)"); axes[1].axis("off")
plt.suptitle(f"환자 {CASE} — 비교", fontsize=13)
plt.tight_layout(); plt.show()

## 9단계. 10명 전체 한눈에 보기 🖼️

마지막으로 10명 모두에 대해 **3D MRI 단면 → pseudo X-ray → 실제 X-ray** 를 한 줄씩 나열합니다.
사람마다 뼈 모양이 조금씩 다른 것을 관찰해 보세요.

In [ ]:
n = len(cases)
fig, axes = plt.subplots(n, 3, figsize=(8, 2.6 * n))
for i, name in enumerate(cases):
    v, r = load_case(name)
    m = v.shape[0] // 2
    axes[i, 0].imshow(np.flipud(v[:, m, :]), cmap="nipy_spectral", vmin=0, vmax=7,
                      interpolation="nearest")
    axes[i, 1].imshow(normalize(make_pseudo_xray(v, mu, 0)) ** 0.6, cmap="gray")
    axes[i, 2].imshow(r, cmap="gray")
    axes[i, 0].set_ylabel(name, fontsize=9)
    for j in range(3):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
axes[0, 0].set_title("3D MRI (단면)")
axes[0, 1].set_title("pseudo X-ray")
axes[0, 2].set_title("실제 X-ray")
plt.tight_layout(); plt.show()

## 🧪 직접 해보기 (도전 과제)

아래를 바꿔 가며 결과가 어떻게 달라지는지 실험해 보세요!

1. **2단계**의 `PICK` 을 0~9로 바꿔 다른 환자 보기
2. **4단계**의 `mu` 에서 연골 값(0.35)을 0.8처럼 키우면 어떻게 될까요?
3. **5단계**에서 더하는 방향을 `axis=1` 대신 `axis=2` 로 바꾸면 옆에서 본 X-ray가 됩니다. 시도해 보세요.
4. **7단계**의 감마 값(0.6)을 0.3 또는 1.5로 바꿔 대비 변화 관찰하기

> 아래 빈 셀에서 자유롭게 코드를 바꿔 실행해 보세요.

In [ ]:
# 여기에서 자유롭게 실험해 보세요!
# 예: 연골도 X선을 많이 막는다면?
mu_test = np.array([0.0, 1.0, 1.0, 1.0, 0.8, 0.8, 0.8, 0.8])
plt.figure(figsize=(4, 6))
plt.imshow(normalize(make_pseudo_xray(vol, mu_test, 0)) ** 0.6, cmap="gray")
plt.title("연골도 많이 막게 바꾼 pseudo X-ray"); plt.axis("off")
plt.show()

## ✅ 정리

- **3D MRI** 는 무릎 속 뼈·연골을 3차원으로 담은 데이터입니다.
- X선은 **단단한 뼈를 많이 막고, 무른 조직은 적게 막습니다.**
- 3D 데이터를 한 방향으로 **쭉 더하면(투영)** → **pseudo X-ray** 가 됩니다.
- 이렇게 만든 pseudo X-ray는 실제 X-ray와 뼈 모양·관절 위치가 닮아 있습니다.

이 원리는 의료 AI 연구에서 **MRI 데이터로 가상의 X-ray를 만들어** 데이터를 늘리거나,
서로 다른 영상을 비교·정렬하는 데 실제로 쓰입니다. 🎓